# Data Preprocessing

## Objective

This notebook cleans the raw datasets so they can be used for machine learning.

# Import libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets

In [2]:
kaggle_df = pd.read_csv("../data/enhanced_menstrual_cycle_dataset.csv")

survey_df = pd.read_csv("../data/participants.csv")

# Dataset Copies

In [3]:
kaggle = kaggle_df.copy()

survey = survey_df.copy()

## Duplicate Removal

Duplicate observations can bias machine learning models by over-representing identical records.

Duplicate rows are identified and removed while keeping the first occurrence.

In [4]:
def remove_duplicates(df, name):

    duplicates = df.duplicated().sum()

    print(f"{name}")
    print(f"Duplicate Rows Before : {duplicates}")

    df = df.drop_duplicates()

    print(f"Rows After Removal : {len(df)}")

    return df

In [5]:
kaggle = remove_duplicates(kaggle, "Kaggle Dataset")

survey = remove_duplicates(survey, "Survey Dataset")

Kaggle Dataset
Duplicate Rows Before : 0
Rows After Removal : 20000
Survey Dataset
Duplicate Rows Before : 0
Rows After Removal : 69


## Detecting Invalid Zero Values

Certain historical menstrual cycle measurements cannot realistically contain only zero values.

Rows where historical measurements consist entirely of zeros are treated as invalid records because they represent missing history rather than actual observations.

These rows are identified and removed before further preprocessing.

In [6]:
print(kaggle.columns.tolist())

['User_ID', 'Age', 'BMI', 'Age_At_Menarche', 'Family_History_Irregular', 'Has_PCOS_Background', 'Smoking_Status', 'Prev_1_Cycle_Length', 'Prev_2_Cycle_Length', 'Prev_3_Cycle_Length', 'Prev_1_Irregular_Flag', 'Prev_2_Irregular_Flag', 'Prev_3_Irregular_Flag', 'Prev_1_Period_Length', 'Prev_2_Period_Length', 'Prev_3_Period_Length', 'Exercise_Frequency', 'Sleep_Hours', 'Diet_Type', 'Stress_Level', 'Alcohol_Consumption', 'Caffeine_Intake', 'Medication_Contraceptive', 'Symptom_Cramps_Current', 'Symptom_Headache_Current', 'Symptom_Fatigue_Current', 'Symptom_Bloating_Current', 'Symptom_Mood_Swings_Current', 'Symptom_Acne_Current', 'Next_Cycle_Length', 'Next_Is_Irregular', 'Next_Period_Length', 'Confidence_Score']


## Removing Identifier Columns

The `User_ID` column uniquely identifies each record but does not contribute to predicting future menstrual cycle characteristics. Therefore, it is removed before model development.

In [7]:
kaggle.drop(columns=["User_ID"], inplace=True)

# Detect Invalid Zero Values & Check for Invalid Rows

In [30]:
history_columns = [
    "Prev_1_Cycle_Length",
    "Prev_2_Cycle_Length",
    "Prev_3_Cycle_Length",
    "Prev_1_Period_Length",
    "Prev_2_Period_Length",
    "Prev_3_Period_Length"
]

invalid_rows = (
    (kaggle[history_columns] == 0)
    .all(axis=1)
)

print(f"Invalid rows found: {invalid_rows.sum()}")

Invalid rows found: 0


# Remove Invalid Rows

In [31]:
kaggle = kaggle.loc[~invalid_rows].reset_index(drop=True)

print("Dataset Shape:", kaggle.shape)

Dataset Shape: (19000, 32)


#  Replace Invalid Zero Values with NaN

In [32]:
invalid_zero_columns = [
    "Age",
    "BMI",
    "Age_At_Menarche",
    "Sleep_Hours",
    "Prev_1_Cycle_Length",
    "Prev_2_Cycle_Length",
    "Prev_3_Cycle_Length",
    "Prev_1_Period_Length",
    "Prev_2_Period_Length",
    "Prev_3_Period_Length"
]

kaggle[invalid_zero_columns] = kaggle[invalid_zero_columns].replace(0, np.nan)

In [12]:
missing = kaggle.isnull().sum()

missing = missing[missing > 0]

display(missing)

Prev_2_Cycle_Length     1000
Prev_3_Cycle_Length     2000
Prev_2_Period_Length    1000
Prev_3_Period_Length    2000
dtype: int64

Since the Kaggle dataset is a synthetically generated dataset, zero values in the historical cycle and period length features were interpreted as unavailable historical records rather than valid measurements. To maintain complete historical sequences for model training, these values were replaced with randomly generated values within clinically plausible ranges (27–32 days for menstrual cycle length and 4–7 days for menstrual period length). A fixed random seed (random_state = 42) was used to ensure reproducibility.

In [13]:
import numpy as np

np.random.seed(42)

# Fill Previous Cycle Lengths

In [14]:
cycle_columns = [
    "Prev_1_Cycle_Length",
    "Prev_2_Cycle_Length",
    "Prev_3_Cycle_Length"
]

for col in cycle_columns:
    mask = kaggle[col] == 0
    kaggle.loc[mask, col] = np.random.randint(27, 33, size=mask.sum())

# Fill Previous Period Lengths

In [15]:
period_columns = [
    "Prev_1_Period_Length",
    "Prev_2_Period_Length",
    "Prev_3_Period_Length"
]

for col in period_columns:
    mask = kaggle[col] == 0
    kaggle.loc[mask, col] = np.random.randint(4, 8, size=mask.sum())

## Missing Value Imputation

Missing values can reduce model performance and prevent certain machine learning algorithms from training effectively.

Numerical features are imputed using the median because it is robust to outliers and preserves the overall distribution of the data better than the mean.

Categorical features are imputed using the most frequent (mode) value to maintain consistency within each category.

# Separate Numerical and Categorical Columns

In [33]:
numerical_columns = kaggle.select_dtypes(include=["number"]).columns
categorical_columns = kaggle.select_dtypes(include=["object", "category"]).columns

# Impute Numerical Features

In [34]:
from sklearn.impute import SimpleImputer

median_imputer = SimpleImputer(strategy="median")

kaggle[numerical_columns] = median_imputer.fit_transform(
    kaggle[numerical_columns]
)

# Verify

In [35]:
print(kaggle.isnull().sum().sum())

0


# 9. Survey Dataset Preprocessing

The survey dataset represents participant information that will eventually be collected through the mobile application.

Unlike the Kaggle dataset, it does not contain the target variables. Therefore, the objective of preprocessing is to ensure that its feature names, data types, and values are consistent with those used during model training.



# Rename Columns

In [20]:
COLUMN_MAPPING = {
    "age": "Age",
    "height": "Height",
    "weight": "Weight",
    "bmi": "BMI",
    "age_at_menarche": "Age_At_Menarche",
    "has_pcos": "Has_PCOS_Background",
    "family_history_irregular": "Family_History_Irregular",
    "medication_contraceptive": "Medication_Contraceptive",
    "sleep_hours": "Sleep_Hours",
    "stress_level": "Stress_Level",
    "exercise_frequency": "Exercise_Frequency",
    "smoking_status": "Smoking_Status",
    "prev1_cycle_length": "Prev_1_Cycle_Length",
    "prev2_cycle_length": "Prev_2_Cycle_Length",
    "prev3_cycle_length": "Prev_3_Cycle_Length",
    "prev1_period_length": "Prev_1_Period_Length",
    "prev2_period_length": "Prev_2_Period_Length",
    "prev3_period_length": "Prev_3_Period_Length",
    "prev1_irregular_flag": "Prev_1_Irregular_Flag",
    "prev2_irregular_flag": "Prev_2_Irregular_Flag",
    "prev3_irregular_flag": "Prev_3_Irregular_Flag"
}

survey.rename(columns=COLUMN_MAPPING, inplace=True)

# Check Column Names

In [21]:
print(survey.columns.tolist())
print(kaggle.columns.tolist())

['participant_id', 'Age', 'Height', 'Weight', 'BMI', 'Age_At_Menarche', 'Has_PCOS_Background', 'Family_History_Irregular', 'Medication_Contraceptive', 'Sleep_Hours', 'Stress_Level', 'Exercise_Frequency', 'Smoking_Status', 'Prev_1_Cycle_Length', 'Prev_2_Cycle_Length', 'Prev_3_Cycle_Length', 'Prev_1_Period_Length', 'Prev_2_Period_Length', 'Prev_3_Period_Length', 'Prev_1_Irregular_Flag', 'Prev_2_Irregular_Flag', 'Prev_3_Irregular_Flag', 'symptom_cramps', 'symptom_fatigue', 'symptom_mood_swings', 'current_cycle_length', 'current_period_length', 'current_irregular_flag', 'created_at']
['Age', 'BMI', 'Age_At_Menarche', 'Family_History_Irregular', 'Has_PCOS_Background', 'Smoking_Status', 'Prev_1_Cycle_Length', 'Prev_2_Cycle_Length', 'Prev_3_Cycle_Length', 'Prev_1_Irregular_Flag', 'Prev_2_Irregular_Flag', 'Prev_3_Irregular_Flag', 'Prev_1_Period_Length', 'Prev_2_Period_Length', 'Prev_3_Period_Length', 'Exercise_Frequency', 'Sleep_Hours', 'Diet_Type', 'Stress_Level', 'Alcohol_Consumption', 'Caff

# Remove Duplicates

In [22]:
duplicates = survey.duplicated().sum()

print(f"Duplicate rows before: {duplicates}")

survey.drop_duplicates(inplace=True)

print(f"Dataset shape: {survey.shape}")

Duplicate rows before: 0
Dataset shape: (69, 29)


# Check Missing Values

In [23]:
missing = survey.isnull().sum()

missing = missing[missing > 0]

display(missing)

Height                     2
Weight                     1
BMI                        2
Sleep_Hours                3
Prev_2_Cycle_Length        1
Prev_3_Cycle_Length        1
Prev_1_Period_Length       1
Prev_2_Period_Length       1
Prev_3_Period_Length       1
Prev_2_Irregular_Flag      1
Prev_3_Irregular_Flag      1
current_cycle_length      68
current_period_length     68
current_irregular_flag    68
dtype: int64

# Handle Missing Values

In [24]:
numerical_columns = survey.select_dtypes(include=["number"]).columns
categorical_columns = survey.select_dtypes(include=["object", "category"]).columns

# Numerical Features

In [25]:
median_imputer = SimpleImputer(strategy="median")

survey[numerical_columns] = median_imputer.fit_transform(
    survey[numerical_columns]
)

# Categorical Features

In [26]:
if len(categorical_columns) > 0:

    mode_imputer = SimpleImputer(strategy="most_frequent")

    survey[categorical_columns] = mode_imputer.fit_transform(
        survey[categorical_columns]
    )

# Standardize Categorical Values

In [27]:
survey = survey.replace({
    "yes": "Yes",
    "YES": "Yes",
    "no": "No",
    "NO": "No"
})

# Check Data Types

In [28]:
survey.info()

<class 'pandas.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   participant_id            69 non-null     str    
 1   Age                       69 non-null     float64
 2   Height                    69 non-null     float64
 3   Weight                    69 non-null     float64
 4   BMI                       69 non-null     float64
 5   Age_At_Menarche           69 non-null     float64
 6   Has_PCOS_Background       69 non-null     str    
 7   Family_History_Irregular  69 non-null     str    
 8   Medication_Contraceptive  69 non-null     str    
 9   Sleep_Hours               69 non-null     float64
 10  Stress_Level              69 non-null     str    
 11  Exercise_Frequency        69 non-null     str    
 12  Smoking_Status            69 non-null     str    
 13  Prev_1_Cycle_Length       69 non-null     float64
 14  Prev_2_Cycle_Length    

# Why the Datasets Are Not Merged

# 10. Data Integration

The Kaggle dataset and the survey dataset are intentionally **not merged**.

The Kaggle dataset is used for training and evaluating the machine learning models because it contains both predictor variables and target variables.

The survey dataset represents user-provided information that will be entered through the mobile application during prediction. Since it does not contain the target variables, it cannot be used for supervised model training.

Instead of merging the datasets, both datasets undergo similar preprocessing procedures independently to ensure that the same feature definitions, naming conventions, and data formats are maintained. This approach allows the trained preprocessing pipeline to be applied consistently during model inference.

# save datasets

In [29]:
kaggle.to_csv("../processed_data/kaggle_cleaned.csv", index=False)

survey.to_csv("../processed_data/survey_cleaned.csv", index=False)